In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    ArrayType
)


# Locations
BLS_RAW_PATH = "/Volumes/bls_dataquest/bronze/bls_dq/raw/bls"
POPULATION_RAW_PATH = "/Volumes/bls_dataquest/bronze/bls_dq/raw/population/population.json"


# Cleanup Function
def clean_column_names(df):
    for old_name in df.columns:

        new_name = (
            old_name
            .strip()
            .replace(" ", "_")
            .replace("\t", "_")
            .replace("\n", "_")
        )

        df = df.withColumnRenamed(old_name, new_name)

    return df


def read_bls_file(file_name):
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "false")
        .load(f"{BLS_RAW_PATH}/{file_name}")
    )

    return clean_column_names(df)


# BLS Data Tables

@dp.table(
    name="pr_data_current",
    comment="Bronze BLS current productivity observations."
)
def pr_data_current():

    return read_bls_file("pr.data.0.Current")


@dp.table(
    name="pr_series",
    comment="Bronze BLS productivity series metadata."
)
def pr_series():

    return read_bls_file("pr.series")


@dp.table(
    name="pr_measure",
    comment="Bronze BLS productivity measure metadata."
)
def pr_measure():

    return read_bls_file("pr.measure")


@dp.table(
    name="pr_period",
    comment="Bronze BLS reporting period metadata."
)
def pr_period():

    return read_bls_file("pr.period")


@dp.table(
    name="pr_sector",
    comment="Bronze BLS sector metadata."
)
def pr_sector():

    return read_bls_file("pr.sector")


@dp.table(
    name="pr_class",
    comment="Bronze BLS productivity classification metadata."
)
def pr_class():

    return read_bls_file("pr.class")


@dp.table(
    name="pr_footnote",
    comment="Bronze BLS productivity footnote metadata."
)
def pr_footnote():

    return read_bls_file("pr.footnote")


@dp.table(
    name="pr_duration",
    comment="Bronze BLS productivity duration metadata."
)
def pr_duration():

    return read_bls_file("pr.duration")


@dp.table(
    name="pr_seasonal",
    comment="Bronze BLS seasonal adjustment metadata."
)
def pr_seasonal():

    return read_bls_file("pr.seasonal")


# Population Table

@dp.table(
    name="population",
    comment="Bronze Data USA annual US population records."
)
def population():

    population_schema = StructType([
        StructField("Nation ID", StringType(), True),
        StructField("Nation", StringType(), True),
        StructField("Year", StringType(), True),
        StructField("Population", DoubleType(), True)
    ])

    response_schema = StructType([
        StructField(
            "data",
            ArrayType(population_schema),
            True
        )
    ])

    raw = (
        spark.read
        .schema(response_schema)
        .json(POPULATION_RAW_PATH)
    )

    return (
        raw
        .select(
            F.explode("data").alias("record")
        )
        .select(
            F.col("record.`Nation ID`").alias("nation_id"),
            F.col("record.Nation").alias("nation"),
            F.col("record.Year").alias("year"),
            F.col("record.Population").alias("population")
        )
    )